In [1]:
# ============================================================
# READ EXACT PR-AUC AND RECALL FROM LATEST SUCCESSFUL
# EVALUATION JOB
# ============================================================

import boto3
import json
from urllib.parse import urlparse

REGION = "ap-southeast-1"

sm_client = boto3.client("sagemaker", region_name=REGION)
s3_client = boto3.client("s3", region_name=REGION)

print("=" * 90)
print("FIND EXACT EVALUATION METRICS")
print("=" * 90)

# ------------------------------------------------------------
# 1. Find recent completed evaluation processing jobs
# ------------------------------------------------------------
jobs = sm_client.list_processing_jobs(
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=100
)["ProcessingJobSummaries"]

evaluation_jobs = [
    j for j in jobs
    if "EvaluateHeartAttack" in j["ProcessingJobName"]
    and j["ProcessingJobStatus"] == "Completed"
]

if not evaluation_jobs:
    raise RuntimeError(
        "No completed EvaluateHeartAttackModel processing job was found."
    )

job_name = evaluation_jobs[0]["ProcessingJobName"]

print("\nLatest completed evaluation job:")
print(job_name)

# ------------------------------------------------------------
# 2. Describe the job and locate the 'evaluation' output
# ------------------------------------------------------------
desc = sm_client.describe_processing_job(
    ProcessingJobName=job_name
)

outputs = desc["ProcessingOutputConfig"]["Outputs"]

evaluation_output = None

for output in outputs:
    if output["OutputName"] == "evaluation":
        evaluation_output = output
        break

if evaluation_output is None:
    raise RuntimeError(
        "The processing job does not contain an output named 'evaluation'."
    )

s3_uri = evaluation_output["S3Output"]["S3Uri"].rstrip("/")

print("\nEvaluation S3 output:")
print(s3_uri)

# ------------------------------------------------------------
# 3. Build evaluation.json S3 location
# ------------------------------------------------------------
evaluation_json_uri = f"{s3_uri}/evaluation.json"

print("\nevaluation.json:")
print(evaluation_json_uri)

parsed = urlparse(evaluation_json_uri)

bucket = parsed.netloc
key = parsed.path.lstrip("/")

# ------------------------------------------------------------
# 4. Download and read evaluation.json
# ------------------------------------------------------------
response = s3_client.get_object(
    Bucket=bucket,
    Key=key
)

evaluation = json.loads(
    response["Body"].read().decode("utf-8")
)

print("\n" + "=" * 90)
print("EVALUATION.JSON CONTENT")
print("=" * 90)

print(
    json.dumps(
        evaluation,
        indent=4
    )
)

# ------------------------------------------------------------
# 5. Extract exact metrics
# ------------------------------------------------------------
recall = evaluation.get("recall")
pr_auc = evaluation.get("pr_auc")

print("\n" + "=" * 90)
print("EXACT QUALITY-GATE METRICS")
print("=" * 90)

print(f"PR-AUC : {pr_auc}")
print(f"Recall : {recall}")

print("\nConfigured quality gate:")
print("PR-AUC >= 0.35")
print("Recall >= 0.75")

# ------------------------------------------------------------
# 6. Verify whether each gate passed
# ------------------------------------------------------------
if pr_auc is not None:
    print(
        f"\nPR-AUC gate : "
        f"{'✅ PASS' if pr_auc >= 0.35 else '❌ FAIL'}"
    )

if recall is not None:
    print(
        f"Recall gate : "
        f"{'✅ PASS' if recall >= 0.75 else '❌ FAIL'}"
    )

print("=" * 90)

FIND EXACT EVALUATION METRICS

Latest completed evaluation job:
pipelines-xldnxhku9k3r-EvaluateHeartAttackM-fg6m79n0lq

Evaluation S3 output:
s3://sagemaker-ap-southeast-1-044528205969/heart-attack-risk/pipeline10/evaluation

evaluation.json:
s3://sagemaker-ap-southeast-1-044528205969/heart-attack-risk/pipeline10/evaluation/evaluation.json



EVALUATION.JSON CONTENT
{
    "threshold": 0.52,
    "accuracy": 0.8002148987007112,
    "precision": 0.1939973375287426,
    "recall": 0.7981080408264875,
    "f1": 0.31212578493890863,
    "roc_auc": 0.8817271887818499,
    "pr_auc": 0.41029027068697754,
    "brier_score": 0.1477919150246493,
    "confusion_matrix": {
        "tn": 53394,
        "fp": 13320,
        "fn": 811,
        "tp": 3206
    },
    "test_rows": 70731,
    "positive_rows": 4017,
    "negative_rows": 66714
}

EXACT QUALITY-GATE METRICS
PR-AUC : 0.41029027068697754
Recall : 0.7981080408264875

Configured quality gate:
PR-AUC >= 0.35
Recall >= 0.75

PR-AUC gate : ✅ PASS
Recall gate : ✅ PASS
